# KG Extraction — Evaluation

Notebook untuk mengevaluasi kualitas Knowledge Graph yang dihasilkan oleh prompt tertentu.

**Alur:**
1. Load test data dari Google Sheets / CSV
2. Run setiap Cypher query ke Neo4j
3. Bandingkan actual vs expected
4. Hitung pass rate per kategori
5. Tulis hasil ke sheet eksperimen

In [13]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

Set `EXPERIMENT_ID` dan sumber test data.

**Experiment ID format:** `{TASK_PREFIX}_{NUMBER}`
- `KGE` = KG Extraction
- `Q2C` = Question to Cypher
- `R2A` = Result to Answer

In [14]:
# === Configuration ===
EXPERIMENT_ID = "KGE_005"   # Unique ID per eksperimen
DOCUMENT_ID = "UU_11_2008"  # Filter test cases untuk dokumen ini (None = semua)

# Sumber test data: 'csv' atau 'gsheets'
TEST_DATA_SOURCE = "gsheets"
CSV_PATH = "data/kg_extraction_test.csv"  # Jika source = csv
GSHEETS_SHEET_NAME = "KG_EXTRACTION_DATA_TEST"  # Jika source = gsheets

# Tulis hasil ke Google Sheets?
WRITE_TO_GSHEETS = True
EXPERIMENT_SHEET_NAME = f"EXP_{EXPERIMENT_ID}"  # 1 sheet per eksperimen

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Document: {DOCUMENT_ID}')
print(f'Test data: {TEST_DATA_SOURCE}')
print(f'Output sheet: {EXPERIMENT_SHEET_NAME}')

Experiment: KGE_005
Document: UU_11_2008
Test data: gsheets
Output sheet: EXP_KGE_005


## Step 1: Load Test Data

In [15]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_SHEET_NAME)
else:
    test_df = pd.read_csv(CSV_PATH)

# Filter by document if specified
if DOCUMENT_ID:
    test_df = test_df[test_df['DOCUMENT_ID'] == DOCUMENT_ID].reset_index(drop=True)

print(f'Loaded {len(test_df)} test cases')
print(f'Categories: {test_df["CATEGORY"].value_counts().to_dict()}')
test_df[['TEST_ID', 'CATEGORY', 'DESCRIPTION', 'COMPARISON_TYPE']].head(10)

2026-04-15 01:13:42,282 - INFO - Retrieving worksheet 'KG_EXTRACTION_DATA_TEST' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-04-15 01:13:43,847 - INFO - Successfully loaded 87 rows from worksheet 'KG_EXTRACTION_DATA_TEST'.


Loaded 87 test cases
Categories: {'definition': 18, 'regulation': 16, 'hierarchy': 15, 'sanction': 13, 'cross_reference': 12, 'subject': 11, 'graph_integrity': 2}


,TEST_ID,CATEGORY,DESCRIPTION,COMPARISON_TYPE
0,UU_11_2008_H01,hierarchy,Bab I (Ketentuan Umum) memuat Pasal 1 dan Pasal 2,CONTAINS_ALL
1,UU_11_2008_H02,hierarchy,Bab II (Asas dan Tujuan) memuat Pasal 3 dan Pa...,CONTAINS_ALL
2,UU_11_2008_H03,hierarchy,Bab III memuat Pasal 5 sampai 12,CONTAINS_ALL
3,UU_11_2008_H04,hierarchy,Bab IV memuat Pasal 13 sampai 16,CONTAINS_ALL
4,UU_11_2008_H05,hierarchy,Bab V memuat Pasal 17 sampai 22,CONTAINS_ALL
5,UU_11_2008_H06,hierarchy,Bab VI memuat Pasal 23 sampai 26,CONTAINS_ALL
6,UU_11_2008_H07,hierarchy,Bab VII memuat Pasal 27 sampai 37,CONTAINS_ALL
7,UU_11_2008_H08,hierarchy,Bab VIII memuat Pasal 38 dan 39,CONTAINS_ALL
8,UU_11_2008_H09,hierarchy,Bab IX memuat Pasal 40 dan 41,CONTAINS_ALL
9,UU_11_2008_H10,hierarchy,Bab X memuat Pasal 42 sampai 44,CONTAINS_ALL


## Step 2: Connect to Neo4j

In [16]:
# === Connect to Neo4j ===
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run('RETURN 1 AS test')
    print(f'✅ Connected to Neo4j ({NEO4J_DATABASE})')

✅ Connected to Neo4j (experiment)


## Step 3: Define Evaluation Logic

In [17]:
def run_cypher(driver, query, database):
    """Run Cypher query and return results as list of dicts."""
    with driver.session(database=database) as session:
        result = session.run(query)
        return [dict(record) for record in result]


def evaluate_test_case(actual_results, expected_values_json, comparison_type):
    """Evaluate a single test case.
    
    Returns:
        dict with keys: passed, score, notes
    """
    expected = json.loads(expected_values_json) if isinstance(expected_values_json, str) else expected_values_json
    
    if comparison_type == 'NOT_EMPTY':
        passed = len(actual_results) > 0
        return {
            'passed': passed,
            'score': 1.0 if passed else 0.0,
            'notes': f'{len(actual_results)} rows returned',
        }
    
    elif comparison_type == 'COUNT_MIN':
        if not actual_results or not expected:
            return {'passed': False, 'score': 0.0, 'notes': 'No results'}
        actual_value = list(actual_results[0].values())[0]
        expected_value = list(expected[0].values())[0]
        passed = actual_value >= expected_value
        return {
            'passed': passed,
            'score': min(1.0, actual_value / expected_value) if expected_value > 0 else 0.0,
            'notes': f'actual={actual_value}, expected_min={expected_value}',
        }
    
    elif comparison_type == 'CONTAINS_ALL':
        if not expected:
            return {'passed': True, 'score': 1.0, 'notes': 'No expected values'}
        
        found = []
        missing = []
        
        for exp_row in expected:
            if exp_row in actual_results:
                found.append(exp_row)
            else:
                missing.append(exp_row)
        
        recall = len(found) / len(expected)
        passed = len(missing) == 0
        
        missing_labels = [str(list(m.values())) for m in missing[:5]]
        notes = f'found={len(found)}/{len(expected)} (recall={recall:.1%})'
        if missing_labels:
            notes += f', missing: {missing_labels}'
        
        return {
            'passed': passed,
            'score': recall,
            'notes': notes,
        }
    
    else:
        return {'passed': False, 'score': 0.0, 'notes': f'Unknown comparison type: {comparison_type}'}


print('✅ Evaluation functions defined')

✅ Evaluation functions defined


## Step 4: Run Evaluation

In [18]:
# === Run All Test Cases ===
results = []

for _, row in test_df.iterrows():
    test_id = row['TEST_ID']
    print(f'Running {test_id}...', end=' ')
    
    try:
        # Run query
        actual = run_cypher(driver, row['CYPHER_QUERY'], NEO4J_DATABASE)
        
        # Evaluate
        eval_result = evaluate_test_case(actual, row['EXPECTED_VALUES'], row['COMPARISON_TYPE'])
        
        status = '✅ PASS' if eval_result['passed'] else '❌ FAIL'
        print(f"{status} — {eval_result['notes']}")
        
        # Format JSON for readability
        try:
            expected_obj = json.loads(row['EXPECTED_VALUES']) if isinstance(row['EXPECTED_VALUES'], str) else row['EXPECTED_VALUES']
            formatted_expected = json.dumps(expected_obj, indent=2, ensure_ascii=False)
        except:
            formatted_expected = row['EXPECTED_VALUES']
            
        formatted_actual = json.dumps(actual, indent=2, ensure_ascii=False)
        
        results.append({
            'TEST_ID': test_id,
            'DOCUMENT_ID': row['DOCUMENT_ID'],
            'CATEGORY': row['CATEGORY'],
            'DESCRIPTION': row['DESCRIPTION'],
            'CYPHER_QUERY': row['CYPHER_QUERY'],
            'EXPECTED_VALUES': row['EXPECTED_VALUES'],
            'FORMATTED_EXPECTED_VALUES': formatted_expected,
            'ACTUAL_VALUES': json.dumps(actual, ensure_ascii=False),
            'FORMATTED_ACTUAL_VALUES': formatted_actual,
            'RESULT': 'PASS' if eval_result['passed'] else 'FAIL',
            'SCORE': round(eval_result['score'], 4),
            'COMPARISON_TYPE': row['COMPARISON_TYPE'],
            'NOTES': eval_result['notes'],
        })
        
    except Exception as e:
        print(f'⚠️ ERROR — {str(e)}')
        
        # Format expected even on error if possible
        try:
            expected_obj = json.loads(row['EXPECTED_VALUES']) if isinstance(row['EXPECTED_VALUES'], str) else row['EXPECTED_VALUES']
            formatted_expected = json.dumps(expected_obj, indent=2, ensure_ascii=False)
        except:
            formatted_expected = row['EXPECTED_VALUES']
            
        results.append({
            'TEST_ID': test_id,
            'DOCUMENT_ID': row['DOCUMENT_ID'],
            'CATEGORY': row['CATEGORY'],
            'DESCRIPTION': row['DESCRIPTION'],
            'CYPHER_QUERY': row['CYPHER_QUERY'],
            'EXPECTED_VALUES': row['EXPECTED_VALUES'],
            'FORMATTED_EXPECTED_VALUES': formatted_expected,
            'ACTUAL_VALUES': '[]',
            'FORMATTED_ACTUAL_VALUES': '[]',
            'RESULT': 'ERROR',
            'SCORE': 0.0,
            'COMPARISON_TYPE': row['COMPARISON_TYPE'],
            'NOTES': str(e),
        })

results_df = pd.DataFrame(results)
print(f'\n=== Done: {len(results)} test cases evaluated ===')

Running UU_11_2008_H01... ❌ FAIL — found=1/2 (recall=50.0%), missing: ["['Pasal 2']"]
Running UU_11_2008_H02... ✅ PASS — found=2/2 (recall=100.0%)
Running UU_11_2008_H03... ❌ FAIL — found=6/8 (recall=75.0%), missing: ["['Pasal 11']", "['Pasal 12']"]
Running UU_11_2008_H04... ❌ FAIL — found=2/4 (recall=50.0%), missing: ["['Pasal 15']", "['Pasal 16']"]
Running UU_11_2008_H05... ✅ PASS — found=6/6 (recall=100.0%)
Running UU_11_2008_H06... ✅ PASS — found=4/4 (recall=100.0%)
Running UU_11_2008_H07... ❌ FAIL — found=5/11 (recall=45.5%), missing: ["['Pasal 32']", "['Pasal 33']", "['Pasal 34']", "['Pasal 35']", "['Pasal 36']"]
Running UU_11_2008_H08... ✅ PASS — found=2/2 (recall=100.0%)
Running UU_11_2008_H09... ✅ PASS — found=2/2 (recall=100.0%)
Running UU_11_2008_H10... ❌ FAIL — found=2/3 (recall=66.7%), missing: ["['Pasal 44']"]
Running UU_11_2008_H11... ❌ FAIL — found=5/8 (recall=62.5%), missing: ["['Pasal 50']", "['Pasal 51']", "['Pasal 52']"]
Running UU_11_2008_H12... ✅ PASS — found=1/1 

## Step 5: Summary

In [19]:
# === Overall Summary ===
total = len(results_df)
passed = len(results_df[results_df['RESULT'] == 'PASS'])
failed = len(results_df[results_df['RESULT'] == 'FAIL'])
errors = len(results_df[results_df['RESULT'] == 'ERROR'])
avg_score = results_df['SCORE'].mean()

print(f'╔══════════════════════════════════════════╗')
print(f'║  KG Extraction Evaluation Report         ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Experiment: {EXPERIMENT_ID:<27s} ║')
print(f'║  Document:   {(DOCUMENT_ID or "ALL"):<27s} ║')
print(f'║  Database:   {NEO4J_DATABASE:<27s} ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Total:  {total:>3d}                              ║')
print(f'║  Pass:   {passed:>3d}  ({passed/total:.1%})                     ║')
print(f'║  Fail:   {failed:>3d}  ({failed/total:.1%})                     ║')
print(f'║  Error:  {errors:>3d}                              ║')
print(f'║  Avg Score: {avg_score:.2%}                       ║')
print(f'╚══════════════════════════════════════════╝')

╔══════════════════════════════════════════╗
║  KG Extraction Evaluation Report         ║
╠══════════════════════════════════════════╣
║  Experiment: KGE_005                     ║
║  Document:   UU_11_2008                  ║
║  Database:   experiment                  ║
╠══════════════════════════════════════════╣
║  Total:   87                              ║
║  Pass:    73  (83.9%)                     ║
║  Fail:    14  (16.1%)                     ║
║  Error:    0                              ║
║  Avg Score: 87.93%                       ║
╚══════════════════════════════════════════╝


In [20]:
# === Per-Category Breakdown ===
print(f'\n{"CATEGORY":<20s} {"PASS":>5s} {"FAIL":>5s} {"TOTAL":>5s} {"RATE":>8s} {"AVG_SCORE":>10s}')
print('─' * 60)

for cat in results_df['CATEGORY'].unique():
    cat_df = results_df[results_df['CATEGORY'] == cat]
    cat_pass = len(cat_df[cat_df['RESULT'] == 'PASS'])
    cat_fail = len(cat_df[cat_df['RESULT'] == 'FAIL'])
    cat_total = len(cat_df)
    cat_rate = cat_pass / cat_total
    cat_avg = cat_df['SCORE'].mean()
    print(f'{cat:<20s} {cat_pass:>5d} {cat_fail:>5d} {cat_total:>5d} {cat_rate:>7.1%} {cat_avg:>9.2%}')

print('─' * 60)
print(f'{"TOTAL":<20s} {passed:>5d} {failed:>5d} {total:>5d} {passed/total:>7.1%} {avg_score:>9.2%}')


CATEGORY              PASS  FAIL TOTAL     RATE  AVG_SCORE
────────────────────────────────────────────────────────────
hierarchy                9     6    15   60.0%    83.31%
definition              13     5    18   72.2%    72.22%
regulation              16     0    16  100.0%   100.00%
sanction                11     2    13   84.6%    84.62%
cross_reference         12     0    12  100.0%   100.00%
subject                 10     1    11   90.9%    90.91%
graph_integrity          2     0     2  100.0%   100.00%
────────────────────────────────────────────────────────────
TOTAL                   73    14    87   83.9%    87.93%


In [21]:
# === Detail: Failed Test Cases ===
failed_df = results_df[results_df['RESULT'] != 'PASS']

if len(failed_df) == 0:
    print('🎉 All test cases passed!')
else:
    print(f'\n❌ Failed/Error test cases ({len(failed_df)}):\n')
    for _, row in failed_df.iterrows():
        print(f"  {row['TEST_ID']} [{row['CATEGORY']}] — {row['DESCRIPTION']}")
        print(f"    Result: {row['RESULT']}, Score: {row['SCORE']:.2%}")
        print(f"    Notes: {row['NOTES']}")
        print()


❌ Failed/Error test cases (14):

  UU_11_2008_H01 [hierarchy] — Bab I (Ketentuan Umum) memuat Pasal 1 dan Pasal 2
    Result: FAIL, Score: 50.00%
    Notes: found=1/2 (recall=50.0%), missing: ["['Pasal 2']"]

  UU_11_2008_H03 [hierarchy] — Bab III memuat Pasal 5 sampai 12
    Result: FAIL, Score: 75.00%
    Notes: found=6/8 (recall=75.0%), missing: ["['Pasal 11']", "['Pasal 12']"]

  UU_11_2008_H04 [hierarchy] — Bab IV memuat Pasal 13 sampai 16
    Result: FAIL, Score: 50.00%
    Notes: found=2/4 (recall=50.0%), missing: ["['Pasal 15']", "['Pasal 16']"]

  UU_11_2008_H07 [hierarchy] — Bab VII memuat Pasal 27 sampai 37
    Result: FAIL, Score: 45.45%
    Notes: found=5/11 (recall=45.5%), missing: ["['Pasal 32']", "['Pasal 33']", "['Pasal 34']", "['Pasal 35']", "['Pasal 36']"]

  UU_11_2008_H10 [hierarchy] — Bab X memuat Pasal 42 sampai 44
    Result: FAIL, Score: 66.67%
    Notes: found=2/3 (recall=66.7%), missing: ["['Pasal 44']"]

  UU_11_2008_H11 [hierarchy] — Bab XI memuat Pasal 45

## Step 6: Save Results

In [22]:
# === Save to CSV ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/kg_eval_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'✅ Results saved to: {output_csv}')

✅ Results saved to: data/evaluation/kg_eval_KGE_005.csv


In [23]:
# === (Optional) Write to Google Sheets ===
if WRITE_TO_GSHEETS:
    from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter

    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')

    writer = GoogleSheetsWriter(
        google_util=gu,
        sheet_id=spreadsheet_id,
        worksheet_name=EXPERIMENT_SHEET_NAME,
        batch_size=5,
    )

    result = writer.write_dataframe(results_df)
    print(f'✅ Written to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    print(f'   Success: {result.successful_rows}, Failed: {result.failed_rows}')
else:
    print('ℹ️  WRITE_TO_GSHEETS = False, skipping Google Sheets upload.')
    print(f'   Set WRITE_TO_GSHEETS = True to write results to sheet "{EXPERIMENT_SHEET_NAME}"')

  0%|          | 0/18 [00:00<?, ?it/s]

2026-04-15 01:13:48,692 - INFO - Successfully wrote row 1/87
2026-04-15 01:13:50,223 - INFO - Successfully wrote row 2/87
2026-04-15 01:13:51,773 - INFO - Successfully wrote row 3/87
2026-04-15 01:13:53,341 - INFO - Successfully wrote row 4/87
2026-04-15 01:13:54,929 - INFO - Successfully wrote row 5/87
  6%|▌         | 1/18 [00:09<02:46,  9.79s/it]2026-04-15 01:13:58,501 - INFO - Successfully wrote row 6/87
2026-04-15 01:14:00,098 - INFO - Successfully wrote row 7/87
2026-04-15 01:14:01,675 - INFO - Successfully wrote row 8/87
2026-04-15 01:14:03,230 - INFO - Successfully wrote row 9/87
2026-04-15 01:14:04,801 - INFO - Successfully wrote row 10/87
 11%|█         | 2/18 [00:19<02:37,  9.84s/it]2026-04-15 01:14:08,768 - INFO - Successfully wrote row 11/87
2026-04-15 01:14:10,361 - INFO - Successfully wrote row 12/87
2026-04-15 01:14:11,926 - INFO - Successfully wrote row 13/87
2026-04-15 01:14:13,552 - INFO - Successfully wrote row 14/87
2026-04-15 01:14:15,106 - INFO - Successfully wro

✅ Written to Google Sheets: EXP_KGE_005
   Success: 87, Failed: 0


In [24]:
# === Cleanup ===
driver.close()
print('✅ Neo4j connection closed.')

✅ Neo4j connection closed.
